In [5]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

import src.features as F
import src.split as S
import src.metrics as M

df = pd.read_parquet("../data/processed/clean.parquet")
feats = F.build_features(df)
train, val, _ = S.time_split(feats)         # test is deliberately not bound to a name

y_tr, y_val = train["duration"], val["duration"]

# 1. constant: train median
pred_const = np.full(len(val), y_tr.median())

# 2. raw upstream estimate; NaN -> train median of the estimate
est_fill = train["estimated_total"].median()
pred_raw = val["estimated_total"].fillna(est_fill)

# 3. calibrated: duration ~ a + b * estimated_total, fit on train only
lin = LinearRegression().fit(train[["estimated_total"]].fillna(est_fill), y_tr)
pred_cal = lin.predict(val[["estimated_total"]].fillna(est_fill))

results = pd.DataFrame({
    "constant (train median)": M.evaluate(y_val, pred_const),
    "raw estimate": M.evaluate(y_val, pred_raw),
    "calibrated estimate": M.evaluate(y_val, pred_cal),
}).T.round(3)
display(results)
print(f"calibrated: duration = {lin.intercept_:.0f} + {lin.coef_[0]:.2f} * estimated_total")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


,mae_min,rmse_min,within_10min,bias_min,over_pct
constant (train median),12.611,17.154,0.505,-3.190,0.483
raw estimate,32.916,36.674,0.026,-32.916,0.000
calibrated estimate,12.248,16.153,0.493,0.197,0.578


calibrated: duration = 1794 + 1.23 * estimated_total


In [6]:
import src.modeling as MD

train_m, (val_m,) = MD.add_store_features(train, [val])

ridge_rows = {}
for alpha in [1, 10, 100, 1000]:
    model = MD.build_ridge(alpha).fit(train_m, y_tr)
    ridge_rows[f"ridge alpha={alpha}"] = M.evaluate(y_val, model.predict(val_m))

results = pd.concat([results, pd.DataFrame(ridge_rows).T.round(3)])
display(results)

,mae_min,rmse_min,within_10min,bias_min,over_pct
constant (train median),12.611,17.154,0.505,-3.190,0.483
raw estimate,32.916,36.674,0.026,-32.916,0.000
calibrated estimate,12.248,16.153,0.493,0.197,0.578
ridge alpha=1,10.069,13.846,0.604,-0.258,0.563
ridge alpha=10,10.068,13.844,0.604,-0.258,0.563
ridge alpha=100,10.062,13.838,0.604,-0.257,0.563
ridge alpha=1000,10.046,13.827,0.605,-0.243,0.565


### Observations: Ridge (validation)

**1. Ridge clearly beats the calibrated baseline.** MAE 10.05 vs 12.25 min (-18%), RMSE 13.83 vs 16.15, and 60.5% of orders within +/-10 min vs 49.3%. Bias is near zero (-0.24 min), so it is not systematically early or late.

**2. Regularization strength is irrelevant here.** alpha from 1 to 1000 moves MAE by only 0.02 min. With ~138k rows and ~100 columns there is little to regularize, so I do not tune it further. **Decision:** alpha = 1000.

**3. Ridge sets the linear ceiling (~10.0 min).** It cannot learn interactions (supply x hour, store x time), which is what LightGBM is for.

In [7]:
cats = MD.fit_categories(train)
X_tr = MD.to_lgbm_frame(train_m, cats)
X_val = MD.to_lgbm_frame(val_m, cats)

lgbm_rows, lgbm_models = {}, {}
for name, obj, extra in [("l2", "regression", {}),
                         ("l1", "regression_l1", {}),
                         ("huber", "huber", {"alpha": 600})]:
    m = MD.fit_lgbm(X_tr, y_tr, X_val, y_val, objective=obj, **extra)
    lgbm_models[name] = m
    lgbm_rows[f"lgbm {name}"] = {**M.evaluate(y_val, m.predict(X_val)), "trees": m.best_iteration_}

results = pd.concat([results, pd.DataFrame(lgbm_rows).T.round(3)])
display(results)

,mae_min,rmse_min,within_10min,bias_min,over_pct,trees
constant (train median),12.611,17.154,0.505,-3.190,0.483,NaN
raw estimate,32.916,36.674,0.026,-32.916,0.000,NaN
calibrated estimate,12.248,16.153,0.493,0.197,0.578,NaN
ridge alpha=1,10.069,13.846,0.604,-0.258,0.563,NaN
ridge alpha=10,10.068,13.844,0.604,-0.258,0.563,NaN
ridge alpha=100,10.062,13.838,0.604,-0.257,0.563,NaN
ridge alpha=1000,10.046,13.827,0.605,-0.243,0.565,NaN
lgbm l2,9.807,13.627,0.624,-0.577,0.552,441.0
lgbm l1,9.679,13.835,0.645,-2.779,0.477,414.0
lgbm huber,9.634,13.704,0.644,-2.241,0.498,578.0


### Observations: LightGBM (validation)

**1. LightGBM beats Ridge, but modestly.** Best MAE 9.63 (Huber) vs 10.05 (Ridge), a 4% gain. Most of the signal is linear, and the remaining error is likely noise from unobserved factors (prep time, traffic, weather).

**2. The objective trades MAE against bias.** L1 and Huber predict the median: lowest MAE (9.68 / 9.63) but they under-predict by 2-3 min on average. L2 predicts the mean: best RMSE (13.63) and smallest bias (-0.58 min) but higher MAE (9.81). **Decision:** Huber (alpha = 600 s) as the primary objective, since it has the lowest MAE with an RMSE close to L2. Bias is reported alongside MAE in every table.

**3. Early stopping used 414-578 trees**, far below the 5,000 cap, so capacity is not the constraint. Validation was used for early stopping, so these scores are slightly optimistic; the test set is used

In [8]:
# (a) Ablation: drop the store encoding
drop = MD.STORE_FEATURES
m_abl = MD.fit_lgbm(X_tr.drop(columns=drop), y_tr, X_val.drop(columns=drop), y_val,
                    objective="huber", alpha=600)
ablation = pd.DataFrame({"lgbm huber, no store features": {
    **M.evaluate(y_val, m_abl.predict(X_val.drop(columns=drop))),
    "trees": m_abl.best_iteration_}}).T.round(3)
display(ablation)

# (b) Random search over LightGBM settings (Huber objective)
space = {
    "num_leaves": [15, 31, 63, 127],
    "min_child_samples": [20, 50, 100, 200],
    "colsample_bytree": [0.5, 0.7, 0.9],
    "reg_lambda": [0.0, 1.0, 10.0],
    "alpha": [300, 600, 1200],
}
rng = np.random.default_rng(42)
fits = []
for _ in range(12):
    cfg = {k: rng.choice(v).item() for k, v in space.items()}
    m = MD.fit_lgbm(X_tr, y_tr, X_val, y_val, objective="huber", **cfg)
    fits.append((cfg, m, M.evaluate(y_val, m.predict(X_val))))
fits.sort(key=lambda t: t[2]["mae_min"])
display(pd.DataFrame([{**cfg, **met, "trees": m.best_iteration_}
                      for cfg, m, met in fits]).round(3))
best_cfg, best_model, _ = fits[0]
print("best config:", best_cfg)

# (c) Blend Ridge and the best LightGBM (weight on LightGBM)
ridge = MD.build_ridge(1000).fit(train_m, y_tr)
p_ridge, p_lgbm = ridge.predict(val_m), best_model.predict(X_val)
blend = {f"w_lgbm={w}": M.evaluate(y_val, w * p_lgbm + (1 - w) * p_ridge)
         for w in [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]}
display(pd.DataFrame(blend).T.round(3))

,mae_min,rmse_min,within_10min,bias_min,over_pct,trees
"lgbm huber, no store features",9.946,14.07,0.627,-2.345,0.499,666.0


,num_leaves,min_child_samples,colsample_bytree,reg_lambda,alpha,mae_min,rmse_min,within_10min,bias_min,over_pct,trees
0,127,20,0.7,1.0,600,9.624,13.681,0.644,-2.183,0.499,497
1,63,50,0.9,1.0,600,9.628,13.700,0.644,-2.231,0.497,575
2,127,100,0.5,10.0,600,9.629,13.698,0.644,-2.251,0.498,379
3,15,200,0.9,1.0,300,9.638,13.767,0.646,-2.642,0.481,3064
4,15,200,0.9,0.0,600,9.640,13.711,0.643,-2.265,0.496,1721
5,15,200,0.7,1.0,600,9.640,13.717,0.644,-2.269,0.495,1736
6,127,20,0.9,0.0,300,9.643,13.770,0.645,-2.606,0.483,733
7,63,50,0.5,10.0,1200,9.677,13.608,0.633,-1.412,0.528,332
8,15,100,0.5,10.0,1200,9.678,13.611,0.636,-1.458,0.526,1208
9,31,20,0.5,1.0,1200,9.681,13.616,0.633,-1.432,0.527,546


best config: {'num_leaves': 127, 'min_child_samples': 20, 'colsample_bytree': 0.7, 'reg_lambda': 1.0, 'alpha': 600}


,mae_min,rmse_min,within_10min,bias_min,over_pct
w_lgbm=0.0,10.046,13.827,0.605,-0.243,0.565
w_lgbm=0.2,9.885,13.701,0.615,-0.631,0.554
w_lgbm=0.4,9.760,13.623,0.625,-1.019,0.542
w_lgbm=0.6,9.672,13.594,0.632,-1.407,0.530
w_lgbm=0.8,9.625,13.613,0.639,-1.795,0.515
w_lgbm=1.0,9.624,13.681,0.644,-2.183,0.499


   **3. Early stopping used 414-578 trees**, far below the 5,000 cap, so capacity is not the constraint. Validation was used for early stopping, so these scores are slightly optimistic; the test set is used once at the end, for the final evaluation.

In [9]:
train, val, test = S.time_split(feats)
print(len(train), len(val), len(test))

138475 28778 30019


In [10]:
import src.pipeline as PL

n_trees = int(round(lgbm_models["huber"].best_iteration_ * PL.TREE_SCALE))
print("final tree count:", n_trees)

train, val, test = S.time_split(feats)          # test is bound for the first and only time
trainval = pd.concat([train, val], ignore_index=True)
y_tv, y_te = trainval["duration"], test["duration"]

# final model, fit on train+val
art = PL.fit_final(trainval, n_trees)
p_final = PL.predict(art, test)

# comparison models, refit on train+val the same way
est_fill = trainval["estimated_total"].median()
p_const = np.full(len(test), y_tv.median())
lin = LinearRegression().fit(trainval[["estimated_total"]].fillna(est_fill), y_tv)
p_cal = lin.predict(test[["estimated_total"]].fillna(est_fill))
tv_m, (te_m,) = MD.add_store_features(trainval, [test])
p_ridge = MD.build_ridge(1000).fit(tv_m, y_tv).predict(te_m)

final_results = pd.DataFrame({
    "constant (median)": M.evaluate(y_te, p_const),
    "calibrated estimate": M.evaluate(y_te, p_cal),
    "ridge": M.evaluate(y_te, p_ridge),
    "lightgbm huber (final)": M.evaluate(y_te, p_final),
}).T.round(3)
display(final_results)

# save, reload, and verify identical predictions
PL.save_artifacts(art, "../models/final.joblib")
art_reloaded = PL.load_artifacts("../models/final.joblib")
assert np.allclose(PL.predict(art_reloaded, test), p_final)
final_results.to_json("../models/test_metrics.json", indent=2)
print("artifacts saved and verified")

final tree count: 694


,mae_min,rmse_min,within_10min,bias_min,over_pct
constant (median),13.838,19.211,0.479,-4.809,0.466
calibrated estimate,13.412,18.149,0.465,-1.535,0.549
ridge,10.953,15.211,0.574,-1.100,0.553
lightgbm huber (final),10.545,15.081,0.609,-3.006,0.481


artifacts saved and verified


### Final evaluation (test set, used once)

All models were refit on train+val. LightGBM used 694 trees (578 from the train-only early-stopping fit, x1.2 for the extra rows). Test = Feb 14-17 (Sat-Tue).

| Model | MAE (min) | RMSE (min) | Within +/-10 min | Bias (min) | Over-predicted |
|---|---|---|---|---|---|
| Constant (median) | 13.84 | 19.21 | 47.9% | -4.81 | 46.6% |
| Calibrated estimate | 13.41 | 18.15 | 46.5% | -1.54 | 54.9% |
| Ridge | 10.95 | 15.21 | 57.4% | -1.10 | 55.3% |
| **LightGBM Huber (final)** | **10.55** | **15.08** | **60.9%** | -3.01 | 48.1% |

**1. The final model holds up.** MAE is 21% below the calibrated baseline on test (13.41 -> 10.55), the same relative gain as on validation (12.25 -> 9.63), so there is no sign of overfitting to validation.

**2. Test is harder than validation for every model** (constant +1.2 min, calibrated +1.2, Ridge +0.9, LightGBM +0.9). The test window is weekend-heavy (mean duration 2,936 s vs 2,831 s on validation), so models should be compared against the baselines, not against validation numbers.

**3. LightGBM beats Ridge by 0.41 min (3.7%)**, the same gap as on validation.

**4. Bias.** The final model under-predicts by 3.0 min on average (48% of predictions are over). The Huber objective targets the median and the test period is slower than the training average. This is a deliberate trade-off for MAE.

**5. Limits.** Four weeks of data, one four-day test window, no confidence intervals yet (planned in Phase 5).